# Social Media Sentiment Analysis — ML Model### Phase 4: Sentiment Classification + Engagement Prediction**Before running:** upload `cleaned_social_media_data.csv` (the file you downloaded after Phase 1) using the folder icon on the left sidebar.This notebook builds two models:1. **Classification** — predict `sentiment_label` (Positive/Neutral/Negative) from the post text2. **Regression** — predict `engagement_score` from post features (followers, hashtags, platform, etc.)

## Step 1: Import libraries

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifier, RandomForestRegressorfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_scorefrom sklearn.metrics import mean_absolute_error, r2_scorefrom sklearn.preprocessing import LabelEncodersns.set_style("whitegrid")

## Step 2: Load the cleaned data

In [ ]:
df = pd.read_csv("cleaned_social_media_data.csv")print("Shape:", df.shape)df.head()

## Part A: Sentiment Classification**Goal:** predict whether a post is Positive, Neutral, or Negative — using only the post text.This shows basic NLP: turning text into numbers (TF-IDF) that a model can learn from.

### Step 3: Prepare text data

In [ ]:
# Drop rows with missing post text just in casetext_df = df.dropna(subset=["post_text", "sentiment_label"]).copy()X_text = text_df["post_text"]y_text = text_df["sentiment_label"]print(y_text.value_counts())

### Step 4: Convert text to numbers using TF-IDFTF-IDF scores each word by how important/unique it is to a post, relative to all posts.

In [ ]:
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")X_tfidf = vectorizer.fit_transform(X_text)print("TF-IDF matrix shape:", X_tfidf.shape)

### Step 5: Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X_tfidf, y_text, test_size=0.2, random_state=42, stratify=y_text)print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

### Step 6: Train a baseline model — Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000)log_model.fit(X_train, y_train)y_pred_log = log_model.predict(X_test)print("Logistic Regression Accuracy:", round(accuracy_score(y_test, y_pred_log), 3))print("\n", classification_report(y_test, y_pred_log))

### Step 7: Try a stronger model — Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)rf_model.fit(X_train, y_train)y_pred_rf = rf_model.predict(X_test)print("Random Forest Accuracy:", round(accuracy_score(y_test, y_pred_rf), 3))print("\n", classification_report(y_test, y_pred_rf))

### Step 8: Interpret the resultIf accuracy is close to ~33-40%, that's expected here — this dataset's sentiment labels likely don't depend heavily onthe exact wording alone (they may be pre-assigned independent of text content). Report this honestly in your README:**"Text-only classification achieved X% accuracy — a baseline that could improve with a larger dataset,n-grams, or a pretrained transformer model."** Being honest about model limits is a strength, not a weakness, in interviews.

In [ ]:
# Confusion matrix — visualize where the model gets confusedcm = confusion_matrix(y_test, y_pred_rf, labels=rf_model.classes_)plt.figure(figsize=(6,5))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",            xticklabels=rf_model.classes_, yticklabels=rf_model.classes_)plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("Confusion Matrix — Sentiment Classification")plt.tight_layout()plt.savefig("confusion_matrix.png", dpi=150)plt.show()

## Part B: Engagement Score Prediction**Goal:** predict `engagement_score` using post features (not text) — followers, hashtags, platform, topic, time posted.This is a regression problem and answers a real business question: **what drives engagement?**

### Step 9: Prepare features

In [ ]:
reg_df = df.dropna(subset=["engagement_score"]).copy()# Encode categorical columns as numberscat_cols = ["platform", "topic_category", "sentiment_label", "day_of_week"]le_dict = {}for col in cat_cols:    le = LabelEncoder()    reg_df[col + "_enc"] = le.fit_transform(reg_df[col].astype(str))    le_dict[col] = lefeature_cols = [    "user_followers_count", "hashtag_count", "mention_count", "post_length",    "posted_hour", "platform_enc", "topic_category_enc", "sentiment_label_enc", "day_of_week_enc"]X_reg = reg_df[feature_cols]y_reg = reg_df["engagement_score"]X_reg.head()

### Step 10: Train/test split and train a Random Forest Regressor

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(    X_reg, y_reg, test_size=0.2, random_state=42)reg_model = RandomForestRegressor(n_estimators=200, random_state=42)reg_model.fit(Xr_train, yr_train)y_pred_reg = reg_model.predict(Xr_test)print("MAE:", round(mean_absolute_error(yr_test, y_pred_reg), 4))print("R2 Score:", round(r2_score(yr_test, y_pred_reg), 4))

### Step 11: Which features matter most?This is the actual business insight — what actually drives engagement.

In [ ]:
importance = pd.Series(reg_model.feature_importances_, index=feature_cols).sort_values(ascending=False)plt.figure(figsize=(8,5))importance.plot(kind="barh", color="#4C72B0")plt.title("Feature Importance — What Drives Engagement Score")plt.xlabel("Importance")plt.gca().invert_yaxis()plt.tight_layout()plt.savefig("feature_importance.png", dpi=150)plt.show()print(importance.round(3))

## Step 12: Download your resultsRun the cell below to download both chart images for your GitHub README.

In [ ]:
from google.colab import filesfiles.download("confusion_matrix.png")files.download("feature_importance.png")

## What's next — writing this upFor your README/resume, summarize:1. **Classification result:** the accuracy you got, and an honest note on what would improve it2. **Regression result:** your R² score, and the **top 2-3 features** that actually drive engagement (from Step 11)3. **One clear takeaway** a business stakeholder would care about — e.g., "Follower count matters more than posting time for driving engagement"This completes all 4 phases of your project: **Python (cleaning/EDA) → SQL → Power BI → ML**.